In [ ]:
%cd ..

In [ ]:

from app.imc2025.prediction import load_from_train


samples = load_from_train("./data")

In [ ]:
from mts.helpers.project.project import Project


last_project_iteration = Project.from_next_iteration("iterations")

In [ ]:
dataset_name = "imc2023_haiper"

In [ ]:
repositories_dirpath = last_project_iteration.iteration_dirpath / "h5_repositories" 

In [ ]:
repositories_dirpath.mkdir(exist_ok=True, parents=True)

In [ ]:
from mts.pipeline.repository import h5 as h5_repo

In [ ]:
image_repository = h5_repo.H5ImageRepository.from_filename(repositories_dirpath, dataset_name)
image_repository.add_repository_metadata(dataset_name=dataset_name)

In [ ]:

for prediction in samples["imc2023_haiper"]:
    image_repository.add_image(prediction.image_filepath)

In [ ]:
import torch
from mts.core.model.mast3r.io import load_model

local_model_directory = (
    "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
)

mast3r_model = load_model(local_model_directory, torch.device("cpu"))

cuda0 = torch.device("cuda:0")

mast3r_model = mast3r_model.to(cuda0) 

In [ ]:
ids = [6, 7, 8, 14]

In [ ]:
from hloc.utils import viz as img_viz

img_viz.plot_images(
    [image_repository.load_image(img_id) for img_id in ids]
)

In [ ]:
ids = list(range(image_repository.images_num()))

In [ ]:
filepaths = [image_repository.get_filepath(img_id) for img_id in ids]

In [ ]:
image_repository._image_id_to_filepath

In [ ]:
import numpy as np

In [ ]:
filepath_map_to_idx = {filepath: num for  num, filepath in enumerate(filepaths)}
idx_to_filepath_map = {num: filepath for  num, filepath in enumerate(filepaths)}
filepaths_as_str = list(map(str, filepaths))
idx_to_filepath_as_str_map = dict(zip(ids, filepaths_as_str))
filepath_as_str_to_ids_map = dict(zip(filepaths_as_str, ids))

filepath_map_to_id = dict(zip(filepaths, ids))

id_to_filepath_map = dict(zip(ids, filepaths))

In [ ]:
import more_itertools as mit

In [ ]:
from mts.core.matcher.dense.mast3r import match_pairs


keypoints_map, matches_map = match_pairs(
    mast3r_model, 
    list(mit.pairwise(range(len(filepaths_as_str)))),
    filepaths_as_str,
    device=cuda0,
)

In [ ]:
local_model_directory = (
    "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
)
retrival_model_dir = "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth"

In [ ]:
scene_graph: str = "retrieval-20-40"

In [ ]:
from mts.pipeline.step.pair.mast3r import Mast3rParer


mast3r_parer = Mast3rParer.from_checkpoints(
    local_model_directory,
    retrival_model_dir,
    scene_graph,
)

In [ ]:
with torch.no_grad():
    similarity_matrix = mast3r_parer.retriever(filepaths_as_str)

In [ ]:
from plotly import express as px

In [ ]:
similarity_matrix.min()

In [ ]:
similarity_matrix = 1 - similarity_matrix

In [ ]:
similarity_matrix = (similarity_matrix - similarity_matrix.min()) / (similarity_matrix.max() - similarity_matrix.min())

In [ ]:
from scipy.sparse.csgraph import minimum_spanning_tree

mst = minimum_spanning_tree(similarity_matrix).toarray()

In [ ]:
import networkx as nx

In [ ]:
def graph_from_similarity(
    matrix,
    labels: list[str],
    threshold: float = 1,
):
    n = matrix.shape[0]
    G = nx.Graph()

    # add nodes
    if labels is None:
        labels = list(range(n))
    G.add_nodes_from(labels)

    # add weighted edges
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i, j] <= threshold:
                G.add_edge(labels[i], labels[j], weight=matrix[i, j])
    return G

In [ ]:
similarity_graph = graph_from_similarity(similarity_matrix, filepaths_as_str, 1.01)

In [ ]:
mst = nx.minimum_spanning_tree(similarity_graph, weight="weight")

In [ ]:
similarity_matrix

In [ ]:
matches_graph = nx.Graph()

In [ ]:
for start_node, end_node in mst.edges:
    pass

In [ ]:
len(list())

In [ ]:
filepath_as_str_to_ids_map

In [ ]:

mst_pairs = [
    (
        filepath_as_str_to_ids_map[st_node],
        filepath_as_str_to_ids_map[nd_node],
    )
    for st_node, nd_node in mst.edges
]

In [ ]:
keypoints_map, matches_map = match_pairs(
    mast3r_model, 
    mst_pairs,
    filepaths_as_str,
    device=cuda0,
)

In [ ]:
list(mit.pairwise(range(len(filepaths_as_str)))),

In [ ]:
list(mst)

In [ ]:

pos = nx.spring_layout(mst, seed=42)

In [ ]:
from plotly import graph_objects as go

In [ ]:
edge_x = []
edge_y = []
edge_text = []

for u, v, d in mst.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    edge_text.append(d.get("weight", ""))

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=2),
    hoverinfo="none",
    mode="lines"
)

In [ ]:
node_x = []
node_y = []
node_text = []

for node in mst.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(str(node))

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="top center",
    hoverinfo="text",
    marker=dict(
        size=20,
        line_width=2
    )
)


In [ ]:
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title="Minimum Spanning Tree",
        showlegend=False,
        hovermode="closest",
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
)

fig.show()


In [ ]:
weights = [d["weight"] for _, _, d in mst.edges(data=True)]
nx.draw(mst, pos, width=weights)

In [ ]:
def mst_from_matrix_nx(
    matrix,
    labels: list[str],
    threshold: float = 0.95,
):
    n = matrix.shape[0]
    G = nx.Graph()

    # add nodes
    if labels is None:
        labels = list(range(n))
    G.add_nodes_from(labels)

    # add weighted edges
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i, j] < threshold:
                G.add_edge(labels[i], labels[j], weight=matrix[i, j])

    # compute MST
    mst = nx.minimum_spanning_tree(G, weight="weight")

    return mst

In [ ]:
np.where(mst > 0)

In [ ]:

px.imshow(similarity_matrix)

In [ ]:
px.imshow(similarity_matrix)

In [ ]:
similarity_matrix[np.arange(len(similarity_matrix)), np.arange(len(similarity_matrix))] = 0

In [ ]:
from app.imc2025.run.from_config import create_imc2025_from_cfg
from omegaconf import OmegaConf
from config.logging import setup, DEBUG

setup(DEBUG)
cfg = OmegaConf.load("config/pipeline/imc2025/0002-pairs.yaml")


In [ ]:
pipeline = create_imc2025_from_cfg(cfg)

In [ ]:
pipeline.run_for("imc2023_haiper")

In [ ]:
dataset_name = "imc2023_haiper"

In [ ]:
pairs_idxs = []
for st_img_id, nd_img_id in pipeline._repositories_map[dataset_name].get_pairs():
    st_filepath = pipeline._repositories_map[dataset_name].get_filepath(st_img_id)
    nd_filepath = pipeline._repositories_map[dataset_name].get_filepath(nd_img_id)
    pairs_idxs.append((filepath_map_to_idx[st_filepath], filepath_map_to_idx[nd_filepath]))


In [ ]:
from pathlib import Path

In [ ]:
import logging


LOGGER = logging.getLogger(__name__)

In [ ]:
from matplotlib import pyplot as plt


def plot_keypoints(kpts, colors="lime", ps=4):
    """Plot keypoints for existing images.
    Args:
        kpts: list of ndarrays of size (N, 2).
        colors: string, or list of list of tuples (one for each keypoints).
        ps: size of the keypoints as float.
    """
    if not isinstance(colors, list):
        colors = [colors] * len(kpts)
    axes = plt.gcf().axes
    for nth, (a, k, c) in enumerate(zip(axes, kpts, colors), 1):
        if len(k) == 0:
            LOGGER.warning("No keypoints for %dth image", nth)
            continue

        a.scatter(k[:, 0], k[:, 1], c=c, s=ps, linewidths=0)

In [ ]:
def map_to_kpts(
    keypoints_map: dict[str, np.ndarray],
    ids: list[int],
    id_to_filepath_map: dict[int, Path],
) -> list[np.ndarray]:
    kpts = []
    for img_id in ids:
        try:
            kp = keypoints_map[str(id_to_filepath_map[img_id])]
        except KeyError:
            kpts.append(np.array([]))
        else:
            kpts.append(kp)
    return kpts

In [ ]:
from hloc.utils import viz as img_viz

kpts = map_to_kpts(keypoints_map, ids, id_to_filepath_map)
img_viz.plot_images(
    [image_repository.load_image(img_id) for img_id in ids],
)
plot_keypoints(kpts)

In [ ]:
from mts.core.matcher.dense.mast3r import extract_dense_keypoints

In [ ]:
import more_itertools as mit
pairwise_idxs = list(mit.pairwise(range(len(filepaths))))
matches_map = extract_dense_keypoints(
    mast3r_model, 
    pairwise_idxs,
    filepaths_as_str,
    device=cuda0,
)

In [ ]:
from scipy.spatial import KDTree
import numpy as np

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from numpy import ma as np_ma

In [ ]:
def match_kpts(
    kpts_intermediary_from: np.ndarray,
    kpts_intermediary_to: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    nd_prev_kpts_tree = KDTree(kpts_intermediary_from)
    distance, indices = nd_prev_kpts_tree.query(kpts_intermediary_to, p=2)

    ma_indices = np_ma.masked_where(distance > 1, indices)

    from_idx, to_idx = np.unique(
        ma_indices,
        return_index=True,
    )
    indices_from = from_idx.compressed()
    indices_to = to_idx[~from_idx.mask]
    return indices_from, indices_to


def match_path(
    kpts_graph: nx.Graph,
    kpts_path: tuple[str, str, str],
    return_kpts: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    start_from, intermediary_from = kpts_path[0], kpts_path[1]
    intermediary_to, end_to = kpts_path[1], kpts_path[2]

    kpts_from, kpts_intermediary_from = np.split(
        kpts_graph[start_from][intermediary_from]["matched_kpts"],
        2,
        axis=1,
    )

    kpts_intermediary_to, kpts_to = np.split(
        kpts_graph[intermediary_to][end_to]["matched_kpts"],
        2,
        axis=1,
    )
    indices_from, indices_to = match_kpts(
        kpts_intermediary_from,
        kpts_intermediary_to,
    )
    if return_kpts:
        return kpts_from, kpts_to, indices_from, indices_to

    return indices_from, indices_to


def merge_path(
    kpts_graph: nx.Graph,
    kpts_path: tuple[str, str, str],
    min_matches: int = 500,
) -> None:
    kpts_from, kpts_to, indices_from, indices_to = match_path(
        kpts_graph,
        kpts_path,
        return_kpts=True,
    )

    if len(indices_from) < min_matches:
        return
    node_from, via, node_to = kpts_path
    matched_kpts = np.concatenate(
        [kpts_from[indices_from], kpts_to[indices_to]], axis=1
    )
    print(f"add new edge ({node_from}, {node_to}) via `{via}`")
    kpts_graph.add_edge(
        node_from,
        node_to,
        matched_kpts=matched_kpts,
    )

In [ ]:
kpts_graph = nx.Graph().to_undirected()
for st_filepath, matched_filepaths_map in matches_map.items():
    for nd_filepath, kpts in matched_filepaths_map.items():
        kpts_graph.add_edge(st_filepath, nd_filepath, matched_kpts=kpts)

In [ ]:
len(kpts_graph.edges)

In [ ]:

nx.draw(kpts_graph, with_labels=True)
plt.show()

In [ ]:
import itertools as it

In [ ]:
pairs = list(it.combinations(filepaths_as_str, 2))

In [ ]:
def do_something(kpts_graph: nx.Graph, pairs: list[tuple[str, str]]):
    for st_fpath, nd_fpath in pairs:
        if not kpts_graph.has_node(st_fpath):
            print(f"`{st_fpath}` not in the graph")
            continue
        if not kpts_graph.has_node(nd_fpath):
            print(f"`{nd_fpath}` not in the graph")
            continue
        st_fpath, nd_fpath = sorted([st_fpath, nd_fpath])
        if not kpts_graph.has_edge(st_fpath, nd_fpath):
            for kpts_path in list(nx.all_simple_paths(kpts_graph, st_fpath, nd_fpath, cutoff=2)):
                # TODO: Add a check if there are enough merges, if so, then merge, otherwise compute the matchings - which is more costly
                merge_path(kpts_graph, kpts_path)

In [ ]:
do_something(kpts_graph, pairs)

In [ ]:

nx.draw(kpts_graph, with_labels=True)
plt.show()

In [ ]:
start_path, middle_path, end_path = (
    "data/train/imc2023_haiper/fountain_image_000.png",
    "data/train/imc2023_haiper/fountain_image_007.png",
    "data/train/imc2023_haiper/fountain_image_012.png",
)

In [ ]:
"data/train/imc2023_haiper/fountain_image_000.png, data/train/imc2023_haiper/fountain_image_012.png", "data/train/imc2023_haiper/fountain_image_166.png"

In [ ]:
len(kpts_graph.edges)

In [ ]:
for st_fpath, nd_fpath in pairs:
    if not kpts_graph.has_edge(st_fpath, nd_fpath):
        break
        

In [ ]:
for kpts_path in list(nx.all_simple_paths(kpts_graph, st_fpath, nd_fpath, cutoff=2)):
    pass

In [ ]:
merge_path(kpts_graph, kpts_path)

In [ ]:
indices_from, indices_to = match_path(kpts_graph, kpts_path)

In [ ]:
start_from, kpts_intermediary_from = kpts_path[0], kpts_path[1]
kpts_intermediary_to, end_to = kpts_path[1], kpts_path[2]

In [ ]:

kpts_from, kpts_intermediary_from = np.split(
    kpts_graph[start_from][kpts_intermediary_from]["matched_kpts"],
    2,
    axis=1,
)

kpts_intermediary_to, kpts_to = np.split(
    kpts_graph[kpts_intermediary_to][end_to]["matched_kpts"],
    2,
    axis=1,
)

In [ ]:
nd_prev_kpts_tree = KDTree(nd_prev_kpts)
distance, indices = nd_prev_kpts_tree.query(st_curr_kpts, p=2)

In [ ]:
kpts_path2

In [ ]:
st_pair_fpaths, nd_pair_fpaths = list(mit.pairwise(kpts_path))

In [ ]:
st_prev_kpts, nd_prev_kpts = np.split(
    kpts_graph[st_pair_fpaths[0]][st_pair_fpaths[1]]["matched_kpts"],
    2,
    axis=1,
)

In [ ]:
st_curr_kpts, nd_curr_kpts = np.split(
    kpts_graph[nd_pair_fpaths[0]][nd_pair_fpaths[1]]["matched_kpts"],
    2,
    axis=1,
)

In [ ]:
nd_prev_kpts_tree = KDTree(nd_prev_kpts)

In [ ]:
distances, indices = nd_prev_kpts_tree.query(st_curr_kpts)

In [ ]:
from numpy import ma

In [ ]:
ma_indices, inverse_indices = np.unique(ma.masked_array( indices, distances > 0), return_index=True, )
unique_indices, inverse_indices = ma_indices.compressed(), inverse_indices[~ma_indices.mask]

mask = np.ones(len(st_curr_kpts), dtype=bool)
mask[inverse_indices] = False

In [ ]:
img_viz.plot_images(
    [
        image_repository.load_image(filepath_map_to_id[Path(st_pair_fpaths[0])]),
        image_repository.load_image(filepath_map_to_id[Path(nd_pair_fpaths[0])]),
        image_repository.load_image(filepath_map_to_id[Path(nd_pair_fpaths[1])]),
    ]
)
# plot_keypoints([st_prev_kpts, nd_prev_kpts, []])
plot_keypoints([st_prev_kpts, nd_prev_kpts, []], colors="blue", ps=4)
plot_keypoints([[], st_curr_kpts, nd_curr_kpts], colors="yellow", ps=3)
plot_keypoints([[], nd_prev_kpts[unique_indices], []], colors="red", ps=1)
plot_keypoints([[], st_curr_kpts[mask], []], colors="lime", ps=1)

In [ ]:
sum(distance <1.5)

In [ ]:
ma_indices = ma.masked_where(distance > 0 , indices)

In [ ]:
from_idx, to_idx = np.unique(
    ma_indices,
    return_index=True,
)

In [ ]:
from_idx.compressed()

In [ ]:
img_viz.plot_images(
    [
        image_repository.load_image(filepath_map_to_id[Path(st_pair_fpaths[0])]),
        image_repository.load_image(filepath_map_to_id[Path(nd_pair_fpaths[1])]),
    ]
)
# plot_keypoints(
#     [st_prev_kpts[from_idx.compressed()], nd_curr_kpts[to_idx[~from_idx.mask]]]
# )
img_viz.plot_matches(
    st_prev_kpts[indices_from],
    nd_curr_kpts[indices_to],
    a=0.1,
)